# MVP de Engenharia de Dados
## Notebook 3 — Camada Bronze

### Objetivo deste notebook

Este notebook é responsável pela ingestão dos dados coletados do Operador Nacional do Sistema Elétrico (ONS) na camada Bronze da arquitetura do projeto.

Os arquivos Parquet correspondentes ao período de 2020 a 2025 foram obtidos no notebook `mvp2-download` a partir do conjunto de dados **Balanço de Energia nos Subsistemas**.

Nesta etapa, os dados serão lidos e persistidos no catálogo do Databricks, preservando sua estrutura próxima ao formato disponibilizado pela fonte.

Não serão realizadas transformações de negócio, padronizações de nomes ou conversões dos atributos numéricos nesta camada. Esses tratamentos serão realizados posteriormente na camada Silver.

### Entrada

- 6 arquivos Parquet do ONS
- Período: 2020 a 2025
- Total validado na coleta: 263.040 registros

### Saída esperada

Os dados serão armazenados como uma tabela da camada Bronze no schema:

`workspace.energia_bronze`

A quantidade de registros da tabela será posteriormente comparada com a quantidade identificada na etapa de coleta, permitindo verificar a preservação dos dados durante a ingestão.


In [0]:
import os

diretorio_atual = os.getcwd()

print("Diretório atual:")
print(diretorio_atual)

print("\nConteúdo do diretório:")
for item in os.listdir(diretorio_atual):
    print(item)

Diretório atual:
/Workspace/Users/irvinglui83@gmail.com

Conteúdo do diretório:
dados_ons
BALANCO_ENERGIA_SUBSISTEMA_2020.parquet
mvp2-download.ipynb
Drafts
mvp6-analise.ipynb
.conversations
MVP_Engenharia_Dados_Energia
mvp3-bronze.ipynb
mvp5-gold.ipynb
.db_internal
mvp4-silver.ipynb


In [0]:
pasta_download = os.path.join(diretorio_atual, "dados_ons")

arquivos_parquet = sorted([
    arquivo
    for arquivo in os.listdir(pasta_download)
    if arquivo.endswith(".parquet")
])

print("Quantidade de arquivos Parquet:", len(arquivos_parquet))

for arquivo in arquivos_parquet:
    print(arquivo)

Quantidade de arquivos Parquet: 6
BALANCO_ENERGIA_SUBSISTEMA_2020.parquet
BALANCO_ENERGIA_SUBSISTEMA_2021.parquet
BALANCO_ENERGIA_SUBSISTEMA_2022.parquet
BALANCO_ENERGIA_SUBSISTEMA_2023.parquet
BALANCO_ENERGIA_SUBSISTEMA_2024.parquet
BALANCO_ENERGIA_SUBSISTEMA_2025.parquet


In [0]:
from pyspark.sql.functions import col

dataframes = []

for ano in range(2020, 2026):
    caminho_ano = f"file:{pasta_download}/BALANCO_ENERGIA_SUBSISTEMA_{ano}.parquet"

    df_ano = spark.read.parquet(caminho_ano)

    df_ano = (
        df_ano
        .withColumn("val_gerhidraulica", col("val_gerhidraulica").cast("string"))
        .withColumn("val_gertermica", col("val_gertermica").cast("string"))
        .withColumn("val_gereolica", col("val_gereolica").cast("string"))
        .withColumn("val_gersolar", col("val_gersolar").cast("string"))
        .withColumn("val_carga", col("val_carga").cast("string"))
        .withColumn("val_intercambio", col("val_intercambio").cast("string"))
    )

    dataframes.append(df_ano)

df_bronze = dataframes[0]

for df in dataframes[1:]:
    df_bronze = df_bronze.unionByName(df)

print("Quantidade de registros da Bronze:", df_bronze.count())
df_bronze.printSchema()

Quantidade de registros da Bronze: 263040
root
 |-- id_subsistema: string (nullable = true)
 |-- nom_subsistema: string (nullable = true)
 |-- din_instante: timestamp (nullable = true)
 |-- val_gerhidraulica: string (nullable = true)
 |-- val_gertermica: string (nullable = true)
 |-- val_gereolica: string (nullable = true)
 |-- val_gersolar: string (nullable = true)
 |-- val_carga: string (nullable = true)
 |-- val_intercambio: string (nullable = true)



In [0]:
for ano in range(2020, 2026):
    caminho_ano = f"file:{pasta_download}/BALANCO_ENERGIA_SUBSISTEMA_{ano}.parquet"
    
    print(f"\n===== SCHEMA {ano} =====")
    
    df_ano = spark.read.parquet(caminho_ano)
    df_ano.printSchema()


===== SCHEMA 2020 =====
root
 |-- id_subsistema: string (nullable = true)
 |-- nom_subsistema: string (nullable = true)
 |-- din_instante: timestamp (nullable = true)
 |-- val_gerhidraulica: string (nullable = true)
 |-- val_gertermica: string (nullable = true)
 |-- val_gereolica: string (nullable = true)
 |-- val_gersolar: string (nullable = true)
 |-- val_carga: string (nullable = true)
 |-- val_intercambio: string (nullable = true)


===== SCHEMA 2021 =====
root
 |-- id_subsistema: string (nullable = true)
 |-- nom_subsistema: string (nullable = true)
 |-- din_instante: timestamp (nullable = true)
 |-- val_gerhidraulica: string (nullable = true)
 |-- val_gertermica: string (nullable = true)
 |-- val_gereolica: string (nullable = true)
 |-- val_gersolar: string (nullable = true)
 |-- val_carga: string (nullable = true)
 |-- val_intercambio: string (nullable = true)


===== SCHEMA 2022 =====
root
 |-- id_subsistema: string (nullable = true)
 |-- nom_subsistema: string (nullable = tru

### Padronização técnica do schema para ingestão na Bronze

Durante a validação dos arquivos anuais foi identificada uma diferença nos tipos de dados fornecidos pela fonte.

Nos arquivos referentes aos anos de **2020, 2021 e 2022**, os atributos de geração, carga e intercâmbio foram armazenados como `string`. Já nos arquivos de **2023, 2024 e 2025**, esses mesmos atributos foram armazenados como `double`.

Essa diferença impediu a leitura e a persistência conjunta dos arquivos utilizando diretamente um único schema inferido pelo Spark.

Para possibilitar a consolidação dos arquivos na camada Bronze, os seguintes atributos foram uniformizados como `string`:

- `val_gerhidraulica`
- `val_gertermica`
- `val_gereolica`
- `val_gersolar`
- `val_carga`
- `val_intercambio`

A escolha por `string` nesta camada busca preservar os valores coletados sem realizar ainda sua interpretação numérica definitiva.

A conversão desses atributos para tipos numéricos adequados será realizada de forma explícita e documentada na camada Silver.

Após a uniformização técnica do schema, os seis arquivos foram consolidados utilizando `unionByName`, resultando em **263.040 registros**, mesma quantidade validada durante a etapa de coleta.


In [0]:
tabela_bronze = "workspace.energia_bronze.balanco_energia_subsistema"

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabela_bronze)

print("Tabela Bronze criada com sucesso:")
print(tabela_bronze)

Tabela Bronze criada com sucesso:
workspace.energia_bronze.balanco_energia_subsistema


In [0]:
df_validacao_bronze = spark.table(
    "workspace.energia_bronze.balanco_energia_subsistema"
)

print("Quantidade de registros na tabela Bronze:", df_validacao_bronze.count())

Quantidade de registros na tabela Bronze: 263040


In [0]:
display(
    spark.table(
        "workspace.energia_bronze.balanco_energia_subsistema"
    ).limit(10)
)

id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio
NE,NORDESTE,2021-01-01T00:00:00.000Z,4433.04700000,2486.07300000,8534.65500000,0E-8,10343.54000000,5110.23499999
N,NORTE,2021-01-01T00:00:00.000Z,4667.04800000,2726.02999999,261.17399999,0E-8,5511.80100000,2142.45100000
SIN,SISTEMA INTERLIGADO NACIONAL,2021-01-01T00:00:00.000Z,34012.19400000,14938.59600000,8946.09700000,0E-8,58414.26000000,-517.37000000
SE,SUDESTE/CENTRO-OESTE,2021-01-01T00:00:00.000Z,23628.33100000,8024.25100000,0E-8,0E-8,33498.35200000,-1845.76999999
S,SUL,2021-01-01T00:00:00.000Z,1283.76800000,1702.24200000,150.26800000,0E-8,9060.56698583,-5924.28898583
NE,NORDESTE,2021-01-01T01:00:00.000Z,3979.40500000,2567.26300000,8566.93800000,0E-8,10428.16700000,4685.43900000
N,NORTE,2021-01-01T01:00:00.000Z,4644.04700000,2697.46999999,198.45500000,0E-8,5470.13300000,2069.83900000
SIN,SISTEMA INTERLIGADO NACIONAL,2021-01-01T01:00:00.000Z,33751.70500000,14964.67000000,8885.14100000,0E-8,58101.50000000,-499.98000000
SE,SUDESTE/CENTRO-OESTE,2021-01-01T01:00:00.000Z,23906.06599999,8001.49199999,0E-8,0E-8,33183.00599999,-1275.44800000
S,SUL,2021-01-01T01:00:00.000Z,1222.18700000,1698.44499999,119.74800000,0E-8,9020.19098754,-5979.81098754


## Resultado da camada Bronze

A ingestão dos dados na camada Bronze foi concluída com sucesso.

Os seis arquivos Parquet correspondentes ao período de 2020 a 2025 foram consolidados e armazenados como uma tabela Delta no catálogo do Databricks:

`workspace.energia_bronze.balanco_energia_subsistema`

Durante a ingestão foi identificada uma diferença de schema entre os arquivos anuais: os atributos relacionados à geração, carga e intercâmbio estavam armazenados como `string` entre 2020 e 2022 e como `double` entre 2023 e 2025.

Para permitir a consolidação dos arquivos e manter os dados próximos à estrutura de origem, esses atributos foram uniformizados como `string` na camada Bronze. A conversão definitiva para tipos numéricos será realizada na camada Silver.

### Validações realizadas

- Quantidade de arquivos de entrada: **6**
- Período: **2020 a 2025**
- Quantidade de atributos: **9**
- Registros identificados na coleta: **263.040**
- Registros persistidos na tabela Bronze: **263.040**
- Diferença de registros entre coleta e Bronze: **0**

A tabela persistida foi novamente consultada diretamente no catálogo e apresentou a mesma quantidade de registros da etapa de coleta, confirmando a preservação dos dados durante a ingestão.

A camada Bronze passa a ser a fonte de entrada para as transformações, padronizações e verificações de qualidade que serão realizadas na camada Silver.